# Old

## BB codes
BB codes are translation-invariant stabilizer codes with qubits on the edges of a $l\times m$ square lattice.
The checks involve 6 qubits, 3 horizontal and 3 vertical, at specific coordinates that are shifted.
The horizontal $Z$ check coordinates are inverses of the vertical $X$ check coordinates.

In [ ]:
def bb_check_list(l, m, horizontal, vertical):
    def triple_to_qubit(xcoord, ycoord, orient):
        return l*m*orient + l*ycoord + xcoord
    check_triples = [[((x+sx) % l,(y+sy) % m,0) for (sx,sy) in horizontal] + [((x+sx) % l,(y+sy) % m,1) for (sx,sy) in vertical]  for x in range(l) for y in range(m)]
    check_list = [[triple_to_qubit(x, y, hv) for x,y,hv in check_triple] for check_triple in check_triples]
    return check_list

## [[98,8,10]] BB code
from [Eberhardt+Steffan 2024](https://arxiv.org/abs/2407.03973)
horizontal: (0,1), (3,0), (4,0)
vertical: (1,0), (0,3), (0,4)

In [ ]:
check_list_overcomplete = bb_check_list(7,7,[(0,1),(3,0),(4,0)], [(1,0),(0,3),(0,4)])

# define logicals
horizontal_logical_pairs = [(0,0),(1,0),(2,0),(5,0),(0,1),(1,1),(2,1),(5,1),(1,2),(3,2),(4,2),(5,2),(1,3),(2,3),(3,3),(6,3),(1,5),(2,5),(3,5),(6,5),(1,6),(3,6),(4,6),(5,6)] # coordinates for a single horizontal X logical
horizontal_logical_list = bb_check_list(7,7,horizontal_logical_pairs, [])
vertical_logical_pairs = [(y,x) for x,y in horizontal_logical_pairs]
vertical_logical_list = bb_check_list(7,7,[],vertical_logical_pairs)
logical_list_overcomplete = horizontal_logical_list + vertical_logical_list

# remove redundant checks and logicals
logicals_overcomplete = tg.transversal_gate_finder.matrix_from_list(98, logical_list_overcomplete)
checks_overcomplete = tg.transversal_gate_finder.matrix_from_list(98, check_list_overcomplete)
independent_check_nrs, independent_logical_nrs = hk.remove_image(checks_overcomplete, logicals_overcomplete)
logical_list = [logical_list_overcomplete[i] for i in independent_logical_nrs]
check_list = [check_list_overcomplete[i] for i in independent_check_nrs]
print(len(independent_check_nrs), "independent checks:\n", independent_check_nrs)
print(len(independent_logical_nrs), "independent logicals:\n", independent_logical_nrs)

# define gate locations: fold-transversal gate
double_gate_tuples = [[(x,y,0),((-1-y)%7,(-1-x)%7,0)] for x in range(7) for y in range(6-x)]
double_gate_tuples += [[(x,y,1),((-1-y)%7,(-1-x)%7,1)] for x in range(7) for y in range(6-x)]
double_gate_list = [[7*7*orient + 7*ycoord + xcoord for xcoord, ycoord, orient in double_gate_tuple] for double_gate_tuple in double_gate_tuples]
print(double_gate_list)
double_gate_locs = np.array(double_gate_list).T
#gate_locs = [all_single_locs(98), double_gate_locs, no_triple_locs()]

46 independent checks:
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
6 independent logicals:
 [0, 1, 2, 49, 50, 51]
[[0, 48], [7, 47], [14, 46], [21, 45], [28, 44], [35, 43], [1, 41], [8, 40], [15, 39], [22, 38], [29, 37], [2, 34], [9, 33], [16, 32], [23, 31], [3, 27], [10, 26], [17, 25], [4, 20], [11, 19], [5, 13], [49, 97], [56, 96], [63, 95], [70, 94], [77, 93], [84, 92], [50, 90], [57, 89], [64, 88], [71, 87], [78, 86], [51, 83], [58, 82], [65, 81], [72, 80], [52, 76], [59, 75], [66, 74], [53, 69], [60, 68], [54, 62]]


In [ ]:
bb134 = tg.transversal_gate_finder(98)
bb134.set_checks_from_list(check_list)
bb134.set_logicals_from_list(logical_list)
bb134.set_ccz_locs_none()
bb134.set_t_locs_all()
bb134.gate_locs[1] = double_gate_locs

z_check_list = bb_check_list(7,7,[(-1,0),(0,-3),(0,-4)], [(0,-1),(-3,0),(-4,0)])
bb134.test_commutation(z_check_list)

X checks and Z checks commute: True
X logicals and Z checks commute: True


In [ ]:
# indeed finds fold-transversal Clifford gate from Eberhardt+Steffan 2024
bb134.find_gates()

Logical generators:
 T^4[1] * 
T^4[0] * T^4[1] * T^4[2] * 
T^4[2] * 
T^4[0] * T^4[3] * T^4[4] * 
T^4[0] * T^4[1] * T^4[5] * 
CS^2[0, 2] * CS^2[1, 2] * CS^2[3, 5] * CS^2[4, 5] * T^4[0] * T^2[1] * T^6[2] * T^4[3] * T^6[4] * T^2[5] * 

Physical representatives:
 T^4[3] * T^4[5] * T^4[7] * T^4[8] * T^4[9] * T^4[13] * T^4[16] * T^4[20] * T^4[23] * T^4[24] * T^4[26] * T^4[27] * T^4[28] * T^4[29] * T^4[31] * T^4[33] * T^4[35] * T^4[36] * T^4[37] * T^4[38] * T^4[40] * T^4[41] * T^4[42] * T^4[43] * 
T^4[3] * T^4[4] * T^4[5] * T^4[6] * T^4[10] * T^4[13] * T^4[14] * T^4[16] * T^4[17] * T^4[20] * T^4[21] * T^4[23] * T^4[25] * T^4[26] * T^4[28] * T^4[30] * T^4[31] * T^4[32] * T^4[33] * T^4[34] * T^4[39] * T^4[40] * T^4[42] * T^4[44] * 
T^4[0] * T^4[3] * T^4[4] * T^4[6] * T^4[8] * T^4[9] * T^4[11] * T^4[13] * T^4[14] * T^4[15] * T^4[16] * T^4[17] * T^4[18] * T^4[20] * T^4[22] * T^4[23] * T^4[32] * T^4[34] * T^4[35] * T^4[36] * T^4[37] * T^4[38] * T^4[42] * T^4[45] * 
T^4[0] * T^4[1] * T^4[3] * T^4[4

In [ ]:
# We only find Pauli logicals without folding for gate locations bounded by stabilizers
bb134.set_cs_locs_checks(check_list)
bb134.find_gates()

Logical generators:
 T^4[2] * 
T^4[0] * 
T^4[0] * T^4[1] * T^4[2] * 
T^4[0] * T^4[1] * T^4[3] * T^4[4] * 
T^4[0] * T^4[1] * T^4[2] * T^4[5] * 
T^4[4] * 

Physical representatives:
 CS^2[35, 88] * CS^2[47, 71] * CS^2[9, 72] * CS^2[42, 96] * CS^2[32, 78] * CS^2[5, 78] * CS^2[29, 54] * CS^2[27, 93] * CS^2[20, 90] * CS^2[35, 78] * CS^2[27, 97] * CS^2[42, 95] * CS^2[20, 67] * CS^2[32, 50] * CS^2[20, 94] * CS^2[5, 68] * CS^2[8, 88] * CS^2[15, 95] * CS^2[42, 67] * CS^2[0, 70] * CS^2[42, 85] * CS^2[20, 66] * CS^2[32, 49] * CS^2[8, 78] * CS^2[0, 92] * CS^2[12, 75] * CS^2[34, 81] * CS^2[15, 85] * CS^2[7, 81] * CS^2[42, 66] * CS^2[20, 56] * CS^2[22, 92] * CS^2[15, 89] * CS^2[34, 80] * CS^2[22, 96] * CS^2[7, 80] * CS^2[42, 56] * CS^2[47, 93] * CS^2[29, 72] * CS^2[0, 54] * CS^2[2, 72] * CS^2[34, 52] * CS^2[34, 70] * CS^2[22, 54] * CS^2[3, 50] * CS^2[32, 95] * CS^2[29, 53] * CS^2[34, 51] * CS^2[5, 72] * CS^2[25, 50] * CS^2[20, 93] * CS^2[22, 53] * CS^2[3, 49] * CS^2[9, 83] * CS^2[35, 67] * CS^2[47, 

In [ ]:
bb134.set_ccz_locs_checks(check_list)
print(bb134.gate_locs[2].shape)
bb134.find_gates()

(3, 920)
Logical generators:
 T^4[2] * 
T^4[0] * 
T^4[0] * T^4[1] * T^4[2] * 
T^4[0] * T^4[1] * T^4[3] * T^4[4] * 
T^4[0] * T^4[1] * T^4[2] * T^4[5] * 
T^4[4] * 

Physical representatives:
 CS^2[35, 88] * CS^2[47, 71] * CS^2[9, 72] * CS^2[42, 96] * CS^2[32, 78] * CS^2[5, 78] * CS^2[29, 54] * CS^2[27, 93] * CS^2[20, 90] * CS^2[35, 78] * CS^2[27, 97] * CS^2[42, 95] * CS^2[20, 67] * CS^2[32, 50] * CS^2[20, 94] * CS^2[5, 68] * CS^2[8, 88] * CS^2[15, 95] * CS^2[42, 67] * CS^2[0, 70] * CS^2[42, 85] * CS^2[20, 66] * CS^2[32, 49] * CS^2[8, 78] * CS^2[0, 92] * CS^2[12, 75] * CS^2[34, 81] * CS^2[15, 85] * CS^2[7, 81] * CS^2[42, 66] * CS^2[20, 56] * CS^2[22, 92] * CS^2[15, 89] * CS^2[34, 80] * CS^2[22, 96] * CS^2[7, 80] * CS^2[42, 56] * CS^2[47, 93] * CS^2[29, 72] * CS^2[0, 54] * CS^2[2, 72] * CS^2[34, 52] * CS^2[34, 70] * CS^2[22, 54] * CS^2[3, 50] * CS^2[32, 95] * CS^2[29, 53] * CS^2[34, 51] * CS^2[5, 72] * CS^2[25, 50] * CS^2[20, 93] * CS^2[22, 53] * CS^2[3, 49] * CS^2[9, 83] * CS^2[35, 67] * 

### BB codes as single 4D code
This doesn't quite work yet (doesn't coincide with 2D definition)

In [ ]:
bb_xcheck_list = [[([0,0,0,0], 0), ([0,0,0,0], 1), ([1,0,0,0], 0), ([0,1,0,0], 1), ([0,0,1,0], 0), ([0,0,0,1], 1)]]
bb_zcheck_list = [[([0,0,0,0], 0), ([0,0,0,0], 1), ([-1,0,0,0], 1), ([0,-1,0,0], 0), ([0,0,-1,0], 1), ([0,0,0,-1], 0)]]

bd_lattice = np.array([[6,0,0,0], [0,12,0,0], [3,-1,0,1], [-1,3,1,0]])
bb_xcheck = generate_ti_check_matrix(2, bb_xcheck_list, bd_lattice)
bb_zcheck = generate_ti_check_matrix(2, bb_zcheck_list, bd_lattice)

print("test x and z stabilizers commute:", ~np.any(bb_xcheck.T @ bb_zcheck % 2 == 1))
zker = hk.z2_kernel_bitgauss(bb_zcheck.T)
ind_xchecks, log_nrs = hk.remove_image(bb_xcheck, zker)
print("independent x checks\n", ind_xchecks)
logs = zker[:, log_nrs]
print(f"{logs.shape[1]} logical x operators\n", logs)

print("indep x checks\n", len(hk.injectify(bb_xcheck)))

test x and z stabilizers commute: True
independent x checks
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71]
0 logical x operators
 []
indep x checks
 72
